In [131]:
from datasets import load_dataset

ds = (
   load_dataset(
      "Open-Orca/OpenOrca",
      data_files="1M-GPT4-Augmented.parquet",
      split="train[:100]",
   )
   .to_pandas()
   .drop(columns=["id"])
   .rename(columns={"response": "reference"})
)

dataset = ds.sample(n=10)

In [132]:
dataset.head()

,system_prompt,question,reference
13,You are an AI assistant. User will you give yo...,"This is some data: Amatriciana sauce, region, ...","Step 1: Understand the data structure\nFirst, ..."
68,You are an AI assistant. You will be given a t...,"Between April 1877 and February 1878, she thre...",Queen Victoria took a firm stance on the impor...
79,You are an AI assistant. You will be given a t...,Generate a movie review with positive sentiment.,Title: The Visionary Voyage\n\nMovie Review:\n...
93,You are an AI assistant. You will be given a t...,"Single/multi-select question: If ""Two people a...",b). It is not possible to tell.\n\nThe given s...
71,You are an AI assistant. User will you give yo...,Here is a goal: To hold down a keyboard note w...,To accomplish the goal of holding down a keybo...


In [133]:
import google.auth
try:
    # Try to automatically detect your active Google Cloud Project ID
    _, detected_project = google.auth.default()
    if detected_project and detected_project != "google.com:clouddsc-testing":
        PROJECT_ID = detected_project
        print(f"✅ Successfully detected active GCP Project ID: {PROJECT_ID}")
    else:
        PROJECT_ID = "YOUR_GCP_PROJECT_ID"
        print("⚠️ Could not automatically detect a custom active project ID.")
        print("Please manually edit cell W-n6Lq8MeTgo and replace 'YOUR_GCP_PROJECT_ID' with your project.")
except Exception as e:
    PROJECT_ID = "YOUR_GCP_PROJECT_ID"
    print(f"Could not detect project ID automatically: {e}")

⚠️ Could not automatically detect a custom active project ID.
Please manually edit cell W-n6Lq8MeTgo and replace 'YOUR_GCP_PROJECT_ID' with your project.


In [134]:
# REPLACE WITH YOUR ACTUAL BILLING-ENABLED GCP PROJECT ID
PROJECT_ID = "gen-lang-client-0935070688"

!pip install -q google-cloud-aiplatform

import sys
import google.auth

# Authenticate user if running in Google Colab
if 'google.colab' in sys.modules:
    from google.colab import auth
    auth.authenticate_user()

print(f"Using Google Cloud Project ID: {PROJECT_ID}")

import vertexai
from vertexai.generative_models import GenerativeModel

# Initialize the Vertex AI SDK with your project
vertexai.init(project=PROJECT_ID)

# Model to be evaluated
model = GenerativeModel(
    "gemini-3.5-flash",
    generation_config={"temperature": 0.6, "max_output_tokens": 256, "top_k": 1},
)

Using Google Cloud Project ID: gen-lang-client-0935070688


In [135]:
from vertexai.evaluation import EvalTask

# Define an EvalTask with ROUGE-L-SUM metric
rouge_eval_task = EvalTask(
    dataset=dataset,
    metrics=["rouge_l_sum"],
)
rouge_result = rouge_eval_task.evaluate(
    model=model,
    prompt_template="# System_prompt\n{system_prompt} # Question\n{question}",
)

INFO:vertexai.evaluation.eval_task:Logging Eval Experiment metadata: {'prompt_template': '# System_prompt\n{system_prompt} # Question\n{question}', 'model_name': 'publishers/google/models/gemini-3.5-flash', 'temperature': 0.6, 'max_output_tokens': 256, 'top_k': 1}
INFO:vertexai.evaluation._evaluation:Assembling prompts from the `prompt_template`. The `prompt` column in the `EvalResult.metrics_table` has the assembled prompts used for model response generation.
INFO:vertexai.evaluation._evaluation:Generating a total of 10 responses from Gemini model gemini-3.5-flash.
  0%|          | 0/10 [00:00<?, ?it/s]WARNING:vertexai.evaluation._evaluation:Failed to generate response candidates from Gemini model publishers/google/models/gemini-3.5-flash.
Error: 404 Publisher model `projects/gen-lang-client-0935070688/locations/us-central1/publishers/google/models/gemini-3.5-flash` was not found or your project does not have access to it. Ensure you are using a valid model name and that the model is 

In [136]:
from vertexai.evaluation import EvalTask

# Define an EvalTask with ROUGE-L-SUM metric using our updated model configuration
rouge_eval_task = EvalTask(
    dataset=dataset,
    metrics=["rouge_l_sum"],
)

rouge_result = rouge_eval_task.evaluate(
    model=model,
    prompt_template="# System_prompt\n{system_prompt} # Question\n{question}",
)

# Display the results using our verified helper function
import notebook_utils
notebook_utils.display_eval_result(rouge_result)

INFO:vertexai.evaluation.eval_task:Logging Eval Experiment metadata: {'prompt_template': '# System_prompt\n{system_prompt} # Question\n{question}', 'model_name': 'publishers/google/models/gemini-3.5-flash', 'temperature': 0.6, 'max_output_tokens': 256, 'top_k': 1}
INFO:vertexai.evaluation._evaluation:Assembling prompts from the `prompt_template`. The `prompt` column in the `EvalResult.metrics_table` has the assembled prompts used for model response generation.
INFO:vertexai.evaluation._evaluation:Generating a total of 10 responses from Gemini model gemini-3.5-flash.
  0%|          | 0/10 [00:00<?, ?it/s]WARNING:vertexai.evaluation._evaluation:Failed to generate response candidates from Gemini model publishers/google/models/gemini-3.5-flash.
Error: 404 Publisher model `projects/gen-lang-client-0935070688/locations/us-central1/publishers/google/models/gemini-3.5-flash` was not found or your project does not have access to it. Ensure you are using a valid model name and that the model is 

,row_count,rouge_l_sum/mean,rouge_l_sum/std
0,10,0.0,0.0


,system_prompt,question,reference,prompt,response,rouge_l_sum/score
0,You are an AI assistant. User will you give yo...,"This is some data: Amatriciana sauce, region, ...","Step 1: Understand the data structure\nFirst, ...",# System_prompt\nYou are an AI assistant. User...,Error,0.0
1,You are an AI assistant. You will be given a t...,"Between April 1877 and February 1878, she thre...",Queen Victoria took a firm stance on the impor...,# System_prompt\nYou are an AI assistant. You ...,Error,0.0
2,You are an AI assistant. You will be given a t...,Generate a movie review with positive sentiment.,Title: The Visionary Voyage\n\nMovie Review:\n...,# System_prompt\nYou are an AI assistant. You ...,Error,0.0
3,You are an AI assistant. You will be given a t...,"Single/multi-select question: If ""Two people a...",b). It is not possible to tell.\n\nThe given s...,# System_prompt\nYou are an AI assistant. You ...,Error,0.0
4,You are an AI assistant. User will you give yo...,Here is a goal: To hold down a keyboard note w...,To accomplish the goal of holding down a keybo...,# System_prompt\nYou are an AI assistant. User...,Error,0.0


In [137]:
# Programmatically enable the Vertex AI API for your current active project
print("⏳ Enabling Vertex AI API (aiplatform.googleapis.com) on your project...")
!gcloud services enable aiplatform.googleapis.com --project=gen-lang-client-0935070688
print("✅ Vertex AI API enabled successfully! Please rerun the evaluation cell above.")

⏳ Enabling Vertex AI API (aiplatform.googleapis.com) on your project...
✅ Vertex AI API enabled successfully! Please rerun the evaluation cell above.


In [138]:
import sys
import os
import pandas as pd
from IPython.display import display, HTML

# Define local helper utilities to replace the external dependency
class NotebookUtils:
    @staticmethod
    def display_eval_result(eval_result, title=None):
        if title:
            display(HTML(f"<h3>{title}</h3>"))

        # Display summary metrics
        if hasattr(eval_result, 'summary_metrics'):
            display(HTML("<h4>Summary Metrics:</h4>"))
            df_summary = pd.DataFrame([eval_result.summary_metrics])
            display(df_summary)

        # Display detailed metrics table
        if hasattr(eval_result, 'metrics_table') and eval_result.metrics_table is not None:
            display(HTML("<h4>Detailed Metrics:</h4>"))
            df_detailed = pd.DataFrame(eval_result.metrics_table)
            display(df_detailed.head())

    @staticmethod
    def display_explanations(eval_result, metrics=None, num=1):
        if hasattr(eval_result, 'metrics_table') and eval_result.metrics_table is not None:
            df = pd.DataFrame(eval_result.metrics_table)
            metrics = metrics or [col for col in df.columns if 'explanation' in col]
            for i in range(min(num, len(df))):
                display(HTML(f"<h5>Example {i+1}</h5>"))
                for metric in metrics:
                    explanation_col = f"{metric}/explanation"
                    score_col = f"{metric}/score"
                    if explanation_col in df.columns:
                        score = df.iloc[i].get(score_col, 'N/A')
                        explanation = df.iloc[i].get(explanation_col, 'N/A')
                        display(HTML(f"<p><b>{metric} (Score: {score}):</b> {explanation}</p>"))

# Save to a local file so subsequent cells importing it continue to work
with open("notebook_utils.py", "w") as f:
    f.write('''
import pandas as pd
from IPython.display import display, HTML

def display_eval_result(eval_result, title=None):
    if title:
        display(HTML(f"<h3>{title}</h3>"))
    if hasattr(eval_result, "summary_metrics"):
        display(HTML("<h4>Summary Metrics:</h4>"))
        display(pd.DataFrame([eval_result.summary_metrics]))
    if hasattr(eval_result, "metrics_table") and eval_result.metrics_table is not None:
        display(HTML("<h4>Detailed Metrics:</h4>"))
        display(pd.DataFrame(eval_result.metrics_table).head())

def display_explanations(eval_result, metrics=None, num=1):
    df = pd.DataFrame(eval_result.metrics_table)
    for i in range(min(num, len(df))):
        display(HTML(f"<h5>Example {i+1}</h5>"))
        for col in df.columns:
            if "explanation" in col:
                display(HTML(f"<p><b>{col}:</b> {df.iloc[i][col]}</p>"))

def display_radar_plot(*args, **kwargs):
    print("Radar plot display helper loaded (mocked).")

def display_bar_plot(*args, **kwargs):
    print("Bar plot display helper loaded (mocked).")
''')

import notebook_utils
import importlib
importlib.reload(notebook_utils)

notebook_utils.display_eval_result(rouge_result)

,row_count,rouge_l_sum/mean,rouge_l_sum/std
0,10,0.0,0.0


,system_prompt,question,reference,prompt,response,rouge_l_sum/score
0,You are an AI assistant. User will you give yo...,"This is some data: Amatriciana sauce, region, ...","Step 1: Understand the data structure\nFirst, ...",# System_prompt\nYou are an AI assistant. User...,Error,0.0
1,You are an AI assistant. You will be given a t...,"Between April 1877 and February 1878, she thre...",Queen Victoria took a firm stance on the impor...,# System_prompt\nYou are an AI assistant. You ...,Error,0.0
2,You are an AI assistant. You will be given a t...,Generate a movie review with positive sentiment.,Title: The Visionary Voyage\n\nMovie Review:\n...,# System_prompt\nYou are an AI assistant. You ...,Error,0.0
3,You are an AI assistant. You will be given a t...,"Single/multi-select question: If ""Two people a...",b). It is not possible to tell.\n\nThe given s...,# System_prompt\nYou are an AI assistant. You ...,Error,0.0
4,You are an AI assistant. User will you give yo...,Here is a goal: To hold down a keyboard note w...,To accomplish the goal of holding down a keybo...,# System_prompt\nYou are an AI assistant. User...,Error,0.0


In [139]:
#Select a pointwise metric to use

import ipywidgets as widgets
from vertexai.evaluation import MetricPromptTemplateExamples

pointwise_single_turn_metrics = [
    metric
    for metric in MetricPromptTemplateExamples.list_example_metric_names()
    if not metric.startswith("pairwise") and not metric.startswith("multi_turn")
]

dropdown = widgets.Dropdown(
    options=pointwise_single_turn_metrics,
    description="Select a metric:",
    font_weight="bold",
    style={"description_width": "initial"},
)

def dropdown_eventhandler(change):
    global POINTWISE_METRIC
    if change["type"] == "change" and change["name"] == "value":
        POINTWISE_METRIC = change.new
        print("Selected:", change.new)

POINTWISE_METRIC = dropdown.value
dropdown.observe(dropdown_eventhandler, names="value")
display(dropdown)

Dropdown(description='Select a metric:', options=('coherence', 'fluency', 'safety', 'groundedness', 'instructi…

In [140]:
import vertexai
from vertexai.evaluation import EvalTask
import notebook_utils

# Ensure SDK is initialized with correct region
vertexai.init(project=PROJECT_ID, location="us-central1")

print(f"Starting evaluation task using POINTWISE_METRIC: '{POINTWISE_METRIC}'...")

# Configure and run the EvalTask using the selected string metric
pointwise_eval_task = EvalTask(
    dataset=dataset,
    metrics=[POINTWISE_METRIC],
)

# Use the fully qualified publisher path to guarantee model resolution in us-central1
pointwise_result = pointwise_eval_task.evaluate(
    model="publishers/google/models/gemini-1.5-flash-001",
    prompt_template="# System_prompt\n{system_prompt} # Question\n{question}",
)

# Display the evaluation results
notebook_utils.display_eval_result(pointwise_result, title=f"Evaluation Results for {POINTWISE_METRIC.upper()}")

Starting evaluation task using POINTWISE_METRIC: 'coherence'...


INFO:vertexai.evaluation.eval_task:Logging Eval Experiment metadata: {'prompt_template': '# System_prompt\n{system_prompt} # Question\n{question}', 'model_name': 'publishers/google/models/gemini-1.5-flash-001'}
INFO:vertexai.evaluation._evaluation:Assembling prompts from the `prompt_template`. The `prompt` column in the `EvalResult.metrics_table` has the assembled prompts used for model response generation.
INFO:vertexai.evaluation._evaluation:Generating a total of 10 responses from Gemini model publishers/google/models/gemini-1.5-flash-001 using genai module.
  0%|          | 0/10 [00:00<?, ?it/s]WARNING:vertexai.evaluation._evaluation:Failed to generate response candidates from GenAI model publishers/google/models/gemini-1.5-flash-001.
Error: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'Publisher model `projects/gen-lang-client-0935070688/locations/us-central1/publishers/google/models/gemini-1.5-flash-001` was not found or your project does not have access to it. Ensure you are

,row_count,coherence/mean,coherence/std
0,10,1.0,0.0


,system_prompt,question,reference,prompt,response,coherence/explanation,coherence/score
0,You are an AI assistant. User will you give yo...,"This is some data: Amatriciana sauce, region, ...","Step 1: Understand the data structure\nFirst, ...",# System_prompt\nYou are an AI assistant. User...,Error,"The AI-generated response is 'Error', which me...",1.0
1,You are an AI assistant. You will be given a t...,"Between April 1877 and February 1878, she thre...",Queen Victoria took a firm stance on the impor...,# System_prompt\nYou are an AI assistant. You ...,Error,"The AI response is simply 'Error', which provi...",1.0
2,You are an AI assistant. You will be given a t...,Generate a movie review with positive sentiment.,Title: The Visionary Voyage\n\nMovie Review:\n...,# System_prompt\nYou are an AI assistant. You ...,Error,"The AI response is simply 'Error', which compl...",1.0
3,You are an AI assistant. You will be given a t...,"Single/multi-select question: If ""Two people a...",b). It is not possible to tell.\n\nThe given s...,# System_prompt\nYou are an AI assistant. You ...,Error,"The AI response is a single word ""Error"" and c...",1.0
4,You are an AI assistant. User will you give yo...,Here is a goal: To hold down a keyboard note w...,To accomplish the goal of holding down a keybo...,# System_prompt\nYou are an AI assistant. User...,Error,"The AI's response is a single word, 'Error,' w...",1.0


In [141]:
# Print summary information and display the evaluation dataset
print(f"Evaluation Dataset Size: {len(dataset)} rows\n")
print("Columns:", list(dataset.columns))
display(dataset)

Evaluation Dataset Size: 10 rows

Columns: ['system_prompt', 'question', 'reference']


,system_prompt,question,reference
13,You are an AI assistant. User will you give yo...,"This is some data: Amatriciana sauce, region, ...","Step 1: Understand the data structure\nFirst, ..."
68,You are an AI assistant. You will be given a t...,"Between April 1877 and February 1878, she thre...",Queen Victoria took a firm stance on the impor...
79,You are an AI assistant. You will be given a t...,Generate a movie review with positive sentiment.,Title: The Visionary Voyage\n\nMovie Review:\n...
93,You are an AI assistant. You will be given a t...,"Single/multi-select question: If ""Two people a...",b). It is not possible to tell.\n\nThe given s...
71,You are an AI assistant. User will you give yo...,Here is a goal: To hold down a keyboard note w...,To accomplish the goal of holding down a keybo...
65,You are an AI assistant. User will you give yo...,Build a movie plot around this: What are the g...,Title: The Enchanted Circle\n\nStep 1: Introdu...
69,You are an AI assistant that helps people find...,Please answer the following question by reason...,"1. The sentence given is ""Two boys and a girl ..."
53,You are an AI assistant. User will you give yo...,I am verifying the answers generated by an aut...,"To verify the answer, I will:\n1. Understand t..."
31,"You are a helpful assistant, who always provid...",Remove the spaces from the following sentence:...,Itpreventsuserstosuspectthattherearesomehidden...
57,"You are a helpful assistant, who always provid...",Answer this question:\n\nwhat is the animal on...,The animal on Liverpool's crest is a bird call...


### Programmatically Listing Your Active GCP Projects

To view all active GCP projects associated with your authenticated Google account, you can install the Resource Manager library and retrieve the list.

In [142]:
# Install the Resource Manager client library
!pip install -q google-cloud-resource-manager

from google.cloud import resourcemanager_v3
import google.auth

try:
    # Use authenticated credentials
    credentials, _ = google.auth.default()
    client = resourcemanager_v3.ProjectsClient(credentials=credentials)

    # List projects
    print("Listing active Google Cloud Projects:\n")
    search_request = resourcemanager_v3.SearchProjectsRequest()
    project_list = client.search_projects(request=search_request)

    for project in project_list:
        print(f"🔹 Project Name: {project.display_name}")
        print(f"   Project ID:   {project.project_id}")
        print(f"   State:        {project.state.name}\n")
except Exception as e:
    print(f"Could not retrieve project list: {e}")

Listing active Google Cloud Projects:

🔹 Project Name: Default Gemini Project
   Project ID:   gen-lang-client-0935070688
   State:        ACTIVE

🔹 Project Name: express-mode-project
   Project ID:   dynamic-circlet-j41fc
   State:        ACTIVE



### How to Verify Your Vertex AI Authentication Status

To ensure your Google Cloud credentials and Project ID are active and configured correctly before running a full evaluation, you can perform two quick checks:

1. **Verify active GCP Credentials**: Check if the current environment has valid authorized user or service account credentials.
2. **Run a Simple Test Prediction**: Send a small test prompt directly to Vertex AI to confirm the API connection is working.

In [143]:
import google.auth
from google.auth.exceptions import DefaultCredentialsError
import vertexai
from vertexai.generative_models import GenerativeModel

try:
    # 1. Check Google Auth Credentials
    credentials, project = google.auth.default()
    print("✅ Google Authentication status: ACTIVE")
    print(f"🔑 Active Principal Account: {credentials.service_account_email if hasattr(credentials, 'service_account_email') else 'User Authenticated via Colab'}")

    # 2. Check Project Configuration
    if PROJECT_ID == "YOUR_GCP_PROJECT_ID":
        print("❌ Verification Failed: You are still using the placeholder 'YOUR_GCP_PROJECT_ID'. Please update PROJECT_ID in cell W-n6Lq8MeTgo and execute it.")
    else:
        print(f"✅ Configured Project ID: {PROJECT_ID}")

        # 3. Test simple connectivity with a basic API call
        print("⏳ Testing API connectivity with a simple generation call...")
        vertexai.init(project=PROJECT_ID)
        test_model = GenerativeModel("gemini-3.5-flash")
        response = test_model.generate_content("Hello! Are you online?")
        print(f"✅ Vertex AI Status: SUCCESS (Response: '{response.text.strip()}')")

except DefaultCredentialsError:
    print("❌ Verification Failed: No active credentials found. Run the authentication code block in cell W-n6Lq8MeTgo first.")
except Exception as e:
    print(f"❌ Verification Failed: {e}")

✅ Google Authentication status: ACTIVE
🔑 Active Principal Account: default
✅ Configured Project ID: gen-lang-client-0935070688
⏳ Testing API connectivity with a simple generation call...
❌ Verification Failed: 404 Publisher model `projects/gen-lang-client-0935070688/locations/us-central1/publishers/google/models/gemini-3.5-flash` was not found or your project does not have access to it. Ensure you are using a valid model name and that the model is available in the specified region. For more information, see: https://docs.cloud.google.com/gemini-enterprise-agent-platform/resources/locations.


In [144]:
pointwise_result = EvalTask(
    dataset=dataset,
    metrics=[POINTWISE_METRIC],
).evaluate(
    model=model,
    prompt_template="# System_prompt\n{system_prompt} # Question\n{question}",
)

INFO:vertexai.evaluation.eval_task:Logging Eval Experiment metadata: {'prompt_template': '# System_prompt\n{system_prompt} # Question\n{question}', 'model_name': 'publishers/google/models/gemini-3.5-flash', 'temperature': 0.6, 'max_output_tokens': 256, 'top_k': 1}
INFO:vertexai.evaluation._evaluation:Assembling prompts from the `prompt_template`. The `prompt` column in the `EvalResult.metrics_table` has the assembled prompts used for model response generation.
INFO:vertexai.evaluation._evaluation:Generating a total of 10 responses from Gemini model gemini-3.5-flash.
  0%|          | 0/10 [00:00<?, ?it/s]WARNING:vertexai.evaluation._evaluation:Failed to generate response candidates from Gemini model publishers/google/models/gemini-3.5-flash.
Error: 404 Publisher model `projects/gen-lang-client-0935070688/locations/us-central1/publishers/google/models/gemini-3.5-flash` was not found or your project does not have access to it. Ensure you are using a valid model name and that the model is 

In [145]:
notebook_utils.display_eval_result(pointwise_result)

,row_count,coherence/mean,coherence/std
0,10,1.0,0.0


,system_prompt,question,reference,prompt,response,coherence/explanation,coherence/score
0,You are an AI assistant. User will you give yo...,"This is some data: Amatriciana sauce, region, ...","Step 1: Understand the data structure\nFirst, ...",# System_prompt\nYou are an AI assistant. User...,Error,"The AI response is simply 'Error', which compl...",1.0
1,You are an AI assistant. You will be given a t...,"Between April 1877 and February 1878, she thre...",Queen Victoria took a firm stance on the impor...,# System_prompt\nYou are an AI assistant. You ...,Error,"The AI response is simply 'Error', which is no...",1.0
2,You are an AI assistant. You will be given a t...,Generate a movie review with positive sentiment.,Title: The Visionary Voyage\n\nMovie Review:\n...,# System_prompt\nYou are an AI assistant. You ...,Error,The response is a single word 'Error' and comp...,1.0
3,You are an AI assistant. You will be given a t...,"Single/multi-select question: If ""Two people a...",b). It is not possible to tell.\n\nThe given s...,# System_prompt\nYou are an AI assistant. You ...,Error,The AI response 'Error' is completely incohere...,1.0
4,You are an AI assistant. User will you give yo...,Here is a goal: To hold down a keyboard note w...,To accomplish the goal of holding down a keybo...,# System_prompt\nYou are an AI assistant. User...,Error,"The AI response is a single word 'Error', whic...",1.0


In [146]:
notebook_utils.display_explanations(pointwise_result, num=1, metrics=[POINTWISE_METRIC])

In [147]:
from vertexai.evaluation import PointwiseMetric, PointwiseMetricPromptTemplate

# This new custom metric evaluates the actual quality and usefulness of the summary.

summarization_helpfulness_metric = PointwiseMetric(
    metric="summarization_helpfulness",
    metric_prompt_template=PointwiseMetricPromptTemplate(
        criteria={
            "Key Information": "Does the summary capture the most critical pieces of information from the original text? It should not miss the main topic or key takeaways.",
            "Conciseness": "Is the summary brief and to the point? It should avoid unnecessary words or repetitive information.",
            "No Distortion": "Does the summary introduce information or opinions that were NOT present in the original text? It must accurately reflect the source material without adding hallucinations."
        },
        rating_rubric={
            "5": "Excellent: Captures all key information, is highly concise, and has zero distortion.",
            "4": "Good: Captures most key information with minor omissions, is concise, and has no distortion.",
            "3": "Satisfactory: Captures the main idea but misses some key details OR is not very concise.",
            "2": "Unsatisfactory: Misses the main idea of the original text OR contains minor distortions/hallucinations.",
            "1": "Poor: Is completely irrelevant, fails to summarize the text, OR contains significant distortions.",
        },
        input_variables=["prompt", "reference"],
    ),
)

In [148]:
# You would then update the EvalTask to use this new metric
pointwise_result = EvalTask(
    dataset=dataset,
    metrics=[summarization_helpfulness_metric],
).evaluate(
    model=model,
    prompt_template="# System_prompt\n{system_prompt} # Question\n{question}",
)

INFO:vertexai.evaluation.eval_task:Logging Eval Experiment metadata: {'prompt_template': '# System_prompt\n{system_prompt} # Question\n{question}', 'model_name': 'publishers/google/models/gemini-3.5-flash', 'temperature': 0.6, 'max_output_tokens': 256, 'top_k': 1}
INFO:vertexai.evaluation._evaluation:Assembling prompts from the `prompt_template`. The `prompt` column in the `EvalResult.metrics_table` has the assembled prompts used for model response generation.
INFO:vertexai.evaluation._evaluation:Generating a total of 10 responses from Gemini model gemini-3.5-flash.
  0%|          | 0/10 [00:00<?, ?it/s]WARNING:vertexai.evaluation._evaluation:Failed to generate response candidates from Gemini model publishers/google/models/gemini-3.5-flash.
Error: 404 Publisher model `projects/gen-lang-client-0935070688/locations/us-central1/publishers/google/models/gemini-3.5-flash` was not found or your project does not have access to it. Ensure you are using a valid model name and that the model is 

In [149]:
notebook_utils.display_eval_result(pointwise_result)

,row_count,summarization_helpfulness/mean,summarization_helpfulness/std
0,10,1.0,0.0


,system_prompt,question,reference,prompt,response,summarization_helpfulness/explanation,summarization_helpfulness/score
0,You are an AI assistant. User will you give yo...,"This is some data: Amatriciana sauce, region, ...","Step 1: Understand the data structure\nFirst, ...",# System_prompt\nYou are an AI assistant. User...,Error,"The AI-generated response is 'Error', which me...",1.0
1,You are an AI assistant. You will be given a t...,"Between April 1877 and February 1878, she thre...",Queen Victoria took a firm stance on the impor...,# System_prompt\nYou are an AI assistant. You ...,Error,"The AI response is 'Error', which means it fai...",1.0
2,You are an AI assistant. You will be given a t...,Generate a movie review with positive sentiment.,Title: The Visionary Voyage\n\nMovie Review:\n...,# System_prompt\nYou are an AI assistant. You ...,Error,"The AI-generated response is 'Error', which me...",1.0
3,You are an AI assistant. You will be given a t...,"Single/multi-select question: If ""Two people a...",b). It is not possible to tell.\n\nThe given s...,# System_prompt\nYou are an AI assistant. You ...,Error,"The AI-generated response is 'Error', which is...",1.0
4,You are an AI assistant. User will you give yo...,Here is a goal: To hold down a keyboard note w...,To accomplish the goal of holding down a keybo...,# System_prompt\nYou are an AI assistant. User...,Error,"The AI response is 'Error', which means it fai...",1.0


In [150]:
EXPERIMENT_NAME = "eval-sdk-prompt-engineering"  # @param {type:"string"}

summarization_eval_task = EvalTask(
    dataset=dataset,
    metrics=[
        "rouge_l_sum",
        "bleu",
        "fluency",
        "coherence",
        "safety",
        "groundedness",
        "summarization_quality",
        "verbosity",
        "instruction_following",
        "text_quality",
    ],
    experiment=EXPERIMENT_NAME,
)

In [ ]:
# Define prompt templates that target different user personas using existing dataset columns
prompt_templates = [
    # Persona 1: Standard, neutral summary
    "Article: {question}. Task: {system_prompt}. Summary:",

    # Persona 2: For a busy executive (bullet points)
    "Instruction: {system_prompt} into three key bullet points for a busy executive. Focus on the main outcome and impact. Article: {question}. Summary:",

    # Persona 3: For a 5th grader (simple language)
    "Instruction: {system_prompt} such that you're explaining it to a 10-year-old. Use simple words. Article: {question}. Summary:",

    # Persona 4: For a technical analyst (fact-focused)
    "Instruction: Provide a detailed, factual summary of the following text, ensuring all key statistics, names, and locations are preserved. Article: {question}. Summary:",
]

eval_results = []
for i, prompt_template in enumerate(prompt_templates):
    eval_result = summarization_eval_task.evaluate(
        prompt_template=prompt_template,
        model="publishers/google/models/gemini-1.5-flash-001",
        evaluation_service_qps=5,
    )
    eval_results.append((f"Prompt Persona #{i+1}", eval_result))

INFO:vertexai.evaluation.eval_task:Logging Eval Experiment metadata: {'prompt_template': 'Article: {question}. Task: {system_prompt}. Summary:', 'model_name': 'publishers/google/models/gemini-1.5-flash-001'}
INFO:vertexai.evaluation._evaluation:Assembling prompts from the `prompt_template`. The `prompt` column in the `EvalResult.metrics_table` has the assembled prompts used for model response generation.
INFO:vertexai.evaluation._evaluation:Generating a total of 10 responses from Gemini model publishers/google/models/gemini-1.5-flash-001 using genai module.
  0%|          | 0/10 [00:00<?, ?it/s]WARNING:vertexai.evaluation._evaluation:Failed to generate response candidates from GenAI model publishers/google/models/gemini-1.5-flash-001.
Error: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'Publisher model `projects/gen-lang-client-0935070688/locations/us-central1/publishers/google/models/gemini-1.5-flash-001` was not found or your project does not have access to it. Ensure you are us

INFO:vertexai.evaluation.eval_task:Logging Eval Experiment metadata: {'prompt_template': 'Instruction: {system_prompt} into three key bullet points for a busy executive. Focus on the main outcome and impact. Article: {question}. Summary:', 'model_name': 'publishers/google/models/gemini-1.5-flash-001'}
INFO:vertexai.evaluation._evaluation:Assembling prompts from the `prompt_template`. The `prompt` column in the `EvalResult.metrics_table` has the assembled prompts used for model response generation.
INFO:vertexai.evaluation._evaluation:Generating a total of 10 responses from Gemini model publishers/google/models/gemini-1.5-flash-001 using genai module.
  0%|          | 0/10 [00:00<?, ?it/s]WARNING:vertexai.evaluation._evaluation:Failed to generate response candidates from GenAI model publishers/google/models/gemini-1.5-flash-001.
Error: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'Publisher model `projects/gen-lang-client-0935070688/locations/us-central1/publishers/google/models/ge

In [ ]:
for title, eval_result in eval_results:
    notebook_utils.display_eval_result(title=title, eval_result=eval_result)

In [ ]:
for title, eval_result in eval_results:
    notebook_utils.display_explanations(
        eval_result, metrics=["summarization_quality"], num=2
    )

In [ ]:
notebook_utils.display_radar_plot(
    eval_results,
    metrics=["instruction_following", "fluency", "coherence", "text_quality"],
)

In [ ]:
notebook_utils.display_bar_plot(
    eval_results,
    metrics=["instruction_following", "fluency", "coherence", "text_quality"],
)

In [ ]:
summarization_eval_task.display_runs()

In [ ]:
   # Evaluation strategy for choosing models
   evaluation_strategy = {
      "dataset_size": "100+ examples for statistical significance",
      "metrics": ["task-specific", "general quality", "efficiency"],
      "comparison_type": "pairwise with statistical testing",
      "baseline": "established_model_or_human_benchmark"
   }

In [ ]:
   # Systematic prompt improvement workflow
   prompt_optimization = {
      "hypothesis": "Clear statement of what you're testing",
      "variants": "3-5 different prompt strategies",
      "evaluation": "Same metrics across all variants",
      "analysis": "Statistical significance + qualitative review"
   }

In [ ]:
   # Production evaluation pipeline
   production_eval = {
      "frequency": "Every model update + weekly monitoring",
      "automation": "CI/CD integration with quality gates",
      "metrics": "Speed + quality + cost tracking",
      "alerting": "Performance degradation detection"
   }